# FinPDF Chunk Evaluation — All 5 Strategies × A1 & B1 Pipelines

## Goal
Find the best (chunking strategy + model) combination for our finance PDF knowledge base.

## Why this matters
S1/S2/S3 use word-based sizing. Finance text tokenizes at 2–4x word ratio.  
**Result: 100% of S1/S2/S3 chunks exceed MiniLM's 256-token limit — model only sees 26–43% of each chunk.**  
S4/S5 fix this with token-exact sizing. This notebook proves whether fixing it actually helps.

## Corpus Reduction Strategy
1 PDF per category (4 total) → ~300-700 chunks per strategy instead of 3,000-7,000.  
Same 4 PDFs used for ALL strategies → fair comparison.

| Category | PDF Selected |
|----------|--------------|
| Regulatory | Regulation E.pdf |
| Consumer Protection | CFPB_consumer_toolkit.pdf |
| Payment Industry | Mastercard Chargeback Guide.pdf |
| Synthetic Policies | chargeback_filing_evidence_policy.pdf |

## What We Test

| Strategy | Type | Chunk Size | Truncation |
|----------|------|-----------|------------|
| S1 | Sliding word-based | 512w / 128w overlap | ~100% truncated |
| S2 | Sliding word-based | 256w / 64w overlap | ~100% truncated |
| S3 | Paragraph word-based | max 512w | ~100% truncated |
| S4 | Token-exact | 200t MiniLM / 400t BGE | 0% truncated |
| S5 | Section-aware + token | 200t MiniLM / 400t BGE | 0% truncated + metadata |

**Pipelines:** A1 (MiniLM Dense + QE + Mistral) and B1 (BGE-Large Dense + QE + Mistral)  
**Evaluation:** Self-retrieval Recall@10 on 20 synthetic questions (5 per category)  
**Results saved to:** `pdf_chunking/Results/chunk_eval_results.json`

## Cell 0 — Repo Root & Config

In [1]:
import sys, os
from pathlib import Path

_root = Path().resolve()
for _ in range(5):
    if (_root / 'config.py').exists():
        break
    _root = _root.parent
os.chdir(_root)
sys.path.insert(0, str(_root))

from config import (
    REGULATORY_PATH, CONSUMER_PROTECTION_PATH,
    PAYMENT_INDUSTRY_PATH, COMPLAINT_PROCEDURES_PATH,
)

CHUNKS_DIR  = os.path.join('Dataset', 'chunks')
RESULTS_DIR = os.path.join('pdf_chunking', 'Results')
os.makedirs(RESULTS_DIR, exist_ok=True)

# ── 1 PDF per category — reduced corpus for fast evaluation ──────────────────
# Strategy: pick mid-sized PDF per category (not too small, not too slow)
# Same 4 files used for ALL 5 strategies → fair comparison
SELECTED_PDFS = {
    'Regulatory':          os.path.join(REGULATORY_PATH,          'Regulation E.pdf'),
    'Consumer_Protection': os.path.join(CONSUMER_PROTECTION_PATH, 'CFPB_consumer_toolkit.pdf'),
    'Payment_Industry':    os.path.join(PAYMENT_INDUSTRY_PATH,    'Mastercard Chargeback Guide.pdf'),
    'Synthetic_Policies':  os.path.join(COMPLAINT_PROCEDURES_PATH,'chargeback_filing_evidence_policy.pdf'),
}

# Existing S1/S2/S3 CSVs (built from ALL PDFs — we will filter to selected 4)
CSV_S1 = os.path.join(CHUNKS_DIR, 'chunks_512w.csv')
CSV_S2 = os.path.join(CHUNKS_DIR, 'chunks_256w.csv')
CSV_S3 = os.path.join(CHUNKS_DIR, 'chunks_paragraph.csv')

print('Repo root:', _root)
print()
print('Selected PDFs for reduced corpus:')
for cat, path in SELECTED_PDFS.items():
    exists = os.path.exists(path)
    size   = int(os.path.getsize(path)/1024) if exists else 0
    print('  [{:>5} KB]  {}  —  {}'.format(size, 'OK' if exists else 'MISSING', cat))

Repo root: /Users/momo/Desktop/GEN AI/Finsearch Project/FinSearch_Intent_Aware_Financial_Document_Intelligence-

Selected PDFs for reduced corpus:
  [ 2774 KB]  OK  —  Regulatory
  [ 2214 KB]  OK  —  Consumer_Protection
  [ 3137 KB]  OK  —  Payment_Industry
  [    8 KB]  OK  —  Synthetic_Policies


## Cell 1 — Install Dependencies

In [2]:
import sys
!{sys.executable} -m pip install -q pdfplumber transformers sentence-transformers faiss-cpu openai python-dotenv

## Cell 2 — Imports

In [3]:
import re, json, time, gc
import pdfplumber
import numpy as np
import pandas as pd
import faiss
from tqdm import tqdm
from transformers import AutoTokenizer
from sentence_transformers import SentenceTransformer
from openai import OpenAI
from dotenv import load_dotenv

os.environ['USE_TF'] = '0'
os.environ['TRANSFORMERS_NO_TF'] = '1'

load_dotenv()
OPENROUTER_API_KEY = os.environ.get('OPENROUTER_API_KEY', '')
if not OPENROUTER_API_KEY:
    raise ValueError('OPENROUTER_API_KEY not found in .env')

# Model names
MINILM_NAME   = 'sentence-transformers/all-MiniLM-L6-v2'
BGELARGE_NAME = 'BAAI/bge-large-en-v1.5'
BGE_PREFIX    = 'Represent this sentence for searching relevant passages: '
GROQ_MODEL    = 'meta-llama/llama-3.3-70b-instruct'
MISTRAL_MODEL = 'mistralai/mistral-large-2411'
RATE_DELAY    = 2.0

# Load MiniLM tokenizer — used for token-exact chunking
tok_minilm = AutoTokenizer.from_pretrained(MINILM_NAME)

print('Imports OK')
print('MiniLM  max tokens: 256  →  target chunk size: 200 tokens')
print('BGE-Large max tokens: 512  →  target chunk size: 400 tokens')

/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Imports OK
MiniLM  max tokens: 256  →  target chunk size: 200 tokens
BGE-Large max tokens: 512  →  target chunk size: 400 tokens


## Cell 3 — Load & Filter S1 / S2 / S3 (Existing Word-Based Chunks)
S1/S2/S3 were already built from all 41 PDFs.  
We filter to keep only the 4 selected PDFs — no re-chunking needed.

In [4]:
selected_filenames = {Path(p).name for p in SELECTED_PDFS.values()}
print('Filtering S1/S2/S3 to selected PDFs:', selected_filenames)
print()

df_s1_full = pd.read_csv(CSV_S1)
df_s2_full = pd.read_csv(CSV_S2)
df_s3_full = pd.read_csv(CSV_S3)

df_s1 = df_s1_full[df_s1_full['source_file'].isin(selected_filenames)].reset_index(drop=True)
df_s2 = df_s2_full[df_s2_full['source_file'].isin(selected_filenames)].reset_index(drop=True)
df_s3 = df_s3_full[df_s3_full['source_file'].isin(selected_filenames)].reset_index(drop=True)

# Compute real token counts (MiniLM tokenizer) to confirm truncation issue
def avg_tokens(df, tokenizer, sample=200):
    sample_texts = df['text'].head(sample).tolist()
    counts = [len(tokenizer.encode(t, add_special_tokens=True)) for t in sample_texts]
    over   = sum(1 for c in counts if c > 256)
    return round(np.mean(counts), 0), round(100 * over / len(counts), 0)

s1_avg, s1_trunc = avg_tokens(df_s1, tok_minilm)
s2_avg, s2_trunc = avg_tokens(df_s2, tok_minilm)
s3_avg, s3_trunc = avg_tokens(df_s3, tok_minilm)

print('Strategy  Chunks  AvgTokens  Truncated%')
print('S1        {:>5}   {:>8.0f}   {:>8.0f}%'.format(len(df_s1), s1_avg, s1_trunc))
print('S2        {:>5}   {:>8.0f}   {:>8.0f}%'.format(len(df_s2), s2_avg, s2_trunc))
print('S3        {:>5}   {:>8.0f}   {:>8.0f}%'.format(len(df_s3), s3_avg, s3_trunc))
print()
print('All S1/S2/S3 chunks exceed MiniLM 256-token limit.')
print('S4/S5 will fix this with token-exact sizing.')

Filtering S1/S2/S3 to selected PDFs: {'Mastercard Chargeback Guide.pdf', 'Regulation E.pdf', 'CFPB_consumer_toolkit.pdf', 'chargeback_filing_evidence_policy.pdf'}



Token indices sequence length is longer than the specified maximum sequence length for this model (855 > 512). Running this sequence through the model will result in indexing errors


Strategy  Chunks  AvgTokens  Truncated%
S1         1431        770        100%
S2         2859        393        100%
S3         1075        764        100%

All S1/S2/S3 chunks exceed MiniLM 256-token limit.
S4/S5 will fix this with token-exact sizing.


## Cell 4 — Extract Selected PDFs (Page by Page)
Page-by-page extraction gives us `page_num` metadata for each chunk.  
Used by both S4 and S5.

In [5]:
def extract_pages(pdf_path):
    """Extract text page by page. Returns list of {page_num, text}."""
    pages = []
    try:
        with pdfplumber.open(pdf_path) as pdf:
            for num, page in enumerate(pdf.pages, 1):
                raw = page.extract_text()
                if raw:
                    clean = re.sub(r'\n{3,}', '\n\n', raw)
                    clean = re.sub(r'[ \t]{2,}', ' ', clean)
                    clean = re.sub(r'[^\x00-\x7F]+', ' ', clean).strip()
                    if len(clean) > 50:
                        pages.append({'page_num': num, 'text': clean})
    except Exception as e:
        print('  WARNING:', pdf_path, '-', e)
    return pages


print('Extracting 4 selected PDFs...')
extracted = {}  # {category: [{page_num, text}]}

for cat, pdf_path in SELECTED_PDFS.items():
    pages = extract_pages(pdf_path)
    extracted[cat] = pages
    words = sum(len(p['text'].split()) for p in pages)
    print('  {:>22}  {:>3} pages  {:>7,} words'.format(cat, len(pages), words))

print('\nExtraction complete.')

Extracting 4 selected PDFs...
              Regulatory  260 pages  130,611 words
     Consumer_Protection  243 pages   61,984 words
        Payment_Industry  974 pages  355,812 words
      Synthetic_Policies    5 pages      633 words

Extraction complete.


## Cell 5 — Build S4: Token-Exact Chunks
Two sizes built in one cell:
- **S4_200**: 200 tokens, 50 overlap — for MiniLM (max 256 tokens)
- **S4_400**: 400 tokens, 100 overlap — for BGE-Large (max 512 tokens)

Both guarantee 0% truncation for their respective models.

In [6]:
def chunk_token_exact(text, tokenizer, target=200, overlap=50):
    """
    Tokenize text → sliding window of `target` tokens with `overlap`.
    Decode each window back to string.
    Guarantees every chunk <= target tokens (verified by construction).
    """
    tokens = tokenizer.encode(text, add_special_tokens=False, truncation=False)
    step   = target - overlap
    chunks = []
    start  = 0
    while start < len(tokens):
        end        = min(start + target, len(tokens))
        chunk_text = tokenizer.decode(tokens[start:end], skip_special_tokens=True).strip()
        if len(chunk_text) > 30:
            chunks.append({'text': chunk_text, 'token_count': end - start})
        start += step
        if end == len(tokens):
            break
    return chunks


def build_s4(extracted_dict, tokenizer, target, overlap, prefix):
    rows = []
    cid  = 0
    for cat, pages in extracted_dict.items():
        src = Path(SELECTED_PDFS[cat]).name
        full_text = '\n\n'.join(p['text'] for p in pages)
        for ch in chunk_token_exact(full_text, tokenizer, target, overlap):
            rows.append({
                'chunk_id':    '{}_{}_{:05d}'.format(prefix, cat[:3].lower(), cid),
                'source_file': src,
                'category':    cat,
                'text':        ch['text'],
                'token_count': ch['token_count'],
                'word_count':  len(ch['text'].split()),
            })
            cid += 1
    return pd.DataFrame(rows)


# S4_200: optimised for MiniLM (256-token limit)
print('Building S4_200 (200t/50 overlap — MiniLM optimised)...')
df_s4_200 = build_s4(extracted, tok_minilm, target=200, overlap=50, prefix='s4_200')
print('  Chunks: {:,}  Avg tokens: {:.0f}  Max: {}'.format(
    len(df_s4_200), df_s4_200['token_count'].mean(), df_s4_200['token_count'].max()))

# S4_400: optimised for BGE-Large (512-token limit)
# BGE-Large uses same WordPiece tokenizer as BERT — approx same token counts
# We reuse tok_minilm for token counting (both are WordPiece-based, close enough)
print('Building S4_400 (400t/100 overlap — BGE-Large optimised)...')
df_s4_400 = build_s4(extracted, tok_minilm, target=400, overlap=100, prefix='s4_400')
print('  Chunks: {:,}  Avg tokens: {:.0f}  Max: {}'.format(
    len(df_s4_400), df_s4_400['token_count'].mean(), df_s4_400['token_count'].max()))

print('\nS4 done. 0% truncation for both models.')

Building S4_200 (200t/50 overlap — MiniLM optimised)...
  Chunks: 5,912  Avg tokens: 200  Max: 200
Building S4_400 (400t/100 overlap — BGE-Large optimised)...
  Chunks: 2,957  Avg tokens: 400  Max: 400

S4 done. 0% truncation for both models.


## Cell 6 — Build S5: Finance Section-Aware Chunks
Detects finance section headers (Risk Factors, Regulatory, Payment, Complaint, etc.)  
Each chunk tagged with `section_title` — never crosses a section boundary.  
Same two sizes as S4: 200t for MiniLM, 400t for BGE-Large.

In [7]:
# Finance section header patterns — covers regulatory, payment, consumer, complaint docs
SECTION_PATTERNS = [
    r'(?i)^(?:SECTION|CHAPTER|ARTICLE|PART)\s+\d+',
    r'(?i)^RISK\s+FACTORS?',
    r'(?i)^FINANCIAL\s+STATEMENTS?',
    r'(?i)^BALANCE\s+SHEET',
    r'(?i)^NOTES?\s+TO\s+(?:CONSOLIDATED\s+)?FINANCIAL',
    r'(?i)^MANAGEMENT.{0,30}DISCUSSION',
    r'(?i)^EXECUTIVE\s+SUMMARY',
    r'(?i)^(?:INTRODUCTION|OVERVIEW|BACKGROUND|SCOPE|PURPOSE)\s*$',
    r'(?i)^REGULATORY\s+(?:FRAMEWORK|REQUIREMENTS?|COMPLIANCE)',
    r'(?i)^COMPLIANCE\s+',
    r'(?i)^PAYMENT\s+(?:SYSTEMS?|PROCESSING|INDUSTRY|SERVICES?)',
    r'(?i)^CONSUMER\s+(?:PROTECTION|RIGHTS?|COMPLAINTS?)',
    r'(?i)^COMPLAINT\s+(?:PROCEDURE|HANDLING|PROCESS)',
    r'(?i)^DISPUTE\s+(?:RESOLUTION|PROCESS|HANDLING)',
    r'(?i)^CHARGEBACK\s+',
    r'(?i)^GOVERNANCE\s+',
    r'(?i)^DATA\s+(?:PROTECTION|PRIVACY|SECURITY)',
    r'(?i)^LIABILITY\s+',
    r'^\d+\.\s+[A-Z][A-Z\s]{4,50}$',
    r'^[A-Z][A-Z\s]{8,60}$',
]


def detect_header(line):
    line = line.strip()
    if len(line) < 4 or len(line) > 80:
        return None
    for pat in SECTION_PATTERNS:
        if re.match(pat, line):
            return line
    return None


def split_sections(text):
    """Split text into sections by header detection. Returns [{section_title, text}]."""
    lines   = text.split('\n')
    secs    = []
    cur_ttl = 'GENERAL'
    buf     = []
    for line in lines:
        h = detect_header(line)
        if h:
            if buf:
                body = ' '.join(buf).strip()
                if len(body) > 50:
                    secs.append({'section_title': cur_ttl, 'text': body})
            cur_ttl = h
            buf = []
        else:
            s = line.strip()
            if s:
                buf.append(s)
    if buf:
        body = ' '.join(buf).strip()
        if len(body) > 50:
            secs.append({'section_title': cur_ttl, 'text': body})
    return secs or [{'section_title': 'GENERAL', 'text': text}]


def build_s5(extracted_dict, tokenizer, target, overlap, prefix):
    rows = []
    cid  = 0
    all_sections = {}
    for cat, pages in extracted_dict.items():
        src = Path(SELECTED_PDFS[cat]).name
        full_text = '\n\n'.join(p['text'] for p in pages)
        sections  = split_sections(full_text)
        for sec in sections:
            title = sec['section_title']
            all_sections[title] = all_sections.get(title, 0) + 1
            for ch in chunk_token_exact(sec['text'], tokenizer, target, overlap):
                rows.append({
                    'chunk_id':      '{}_{}_{:05d}'.format(prefix, cat[:3].lower(), cid),
                    'source_file':   src,
                    'category':      cat,
                    'section_title': title,
                    'text':          ch['text'],
                    'token_count':   ch['token_count'],
                    'word_count':    len(ch['text'].split()),
                })
                cid += 1
    return pd.DataFrame(rows), all_sections


print('Building S5_200 (section-aware, 200t — MiniLM)...')
df_s5_200, sections_200 = build_s5(extracted, tok_minilm, target=200, overlap=50,  prefix='s5_200')
print('  Chunks: {:,}  Sections detected: {}  Max tokens: {}'.format(
    len(df_s5_200), df_s5_200['section_title'].nunique(), df_s5_200['token_count'].max()))

print('Building S5_400 (section-aware, 400t — BGE-Large)...')
df_s5_400, sections_400 = build_s5(extracted, tok_minilm, target=400, overlap=100, prefix='s5_400')
print('  Chunks: {:,}  Sections detected: {}  Max tokens: {}'.format(
    len(df_s5_400), df_s5_400['section_title'].nunique(), df_s5_400['token_count'].max()))

print('\nTop sections detected across all 4 PDFs:')
for title, count in sorted(sections_200.items(), key=lambda x: x[1], reverse=True)[:10]:
    print('  {:>3}x  {}'.format(count, title[:60]))

Building S5_200 (section-aware, 200t — MiniLM)...
  Chunks: 6,153  Sections detected: 1154  Max tokens: 200
Building S5_400 (section-aware, 400t — BGE-Large)...
  Chunks: 3,368  Sections detected: 1154  Max tokens: 400

Top sections detected across all 4 PDFs:
   47x  Chargeback Remedied)
   45x  Compliance Case Filing
   33x  GETTING STARTED
   19x  chargeback before using this second presentment.
   18x  Compliance Case
   13x  compliance case filing date.
   10x  Introduction
    9x  Chargeback rules on this subject appear in the  Domestic Cha
    9x  compliance case number of the original pre-compliance case.
    8x  MULTIPLE SESSIONS


## Cell 7 — All 5 Strategies: Chunk Stats Comparison

In [8]:
# For A1 (MiniLM): S1, S2, S3, S4_200, S5_200
# For B1 (BGE-Large): S1, S2, S3, S4_400, S5_400

minilm_strategies = [
    ('S1', 'Sliding-512w',     df_s1,     'word',  512, 100.0),
    ('S2', 'Sliding-256w',     df_s2,     'word',  256, 100.0),
    ('S3', 'Paragraph',        df_s3,     'word',  512, 100.0),
    ('S4', 'Token-Exact-200',  df_s4_200, 'token', 200,   0.0),
    ('S5', 'Section-200',      df_s5_200, 'token', 200,   0.0),
]

bge_strategies = [
    ('S1', 'Sliding-512w',     df_s1,     'word',  512, 100.0),
    ('S2', 'Sliding-256w',     df_s2,     'word',  256, 100.0),
    ('S3', 'Paragraph',        df_s3,     'word',  512, 100.0),
    ('S4', 'Token-Exact-400',  df_s4_400, 'token', 400,   0.0),
    ('S5', 'Section-400',      df_s5_400, 'token', 400,   0.0),
]

def print_stats(label, strategies, tokenizer):
    print('\n{}'.format(label))
    print('=' * 72)
    print('  {:<4} {:<20} {:>7} {:>6} {:>10} {:>10}'.format(
        'ID', 'Strategy', 'Chunks', 'Unit', 'AvgTokens', 'Trunc%'))
    print('-' * 72)
    for sid, name, df, unit, target, trunc in strategies:
        if 'token_count' in df.columns:
            avg_tok = df['token_count'].mean()
        else:
            sample  = df['text'].head(100).tolist()
            counts  = [len(tokenizer.encode(t, add_special_tokens=True)) for t in sample]
            avg_tok = np.mean(counts)
        tag = '  <- TRUNCATED' if trunc > 0 else '  <- OK'
        print('  {:<4} {:<20} {:>7,} {:>6} {:>10.0f} {:>9.0f}%{}'.format(
            sid, name, len(df), unit, avg_tok, trunc, tag))
    print('=' * 72)

print_stats('A1 Pipeline (MiniLM — 256 token limit)', minilm_strategies, tok_minilm)
print_stats('B1 Pipeline (BGE-Large — 512 token limit)', bge_strategies, tok_minilm)


A1 Pipeline (MiniLM — 256 token limit)
  ID   Strategy              Chunks   Unit  AvgTokens     Trunc%
------------------------------------------------------------------------
  S1   Sliding-512w           1,431   word        781       100%  <- TRUNCATED
  S2   Sliding-256w           2,859   word        386       100%  <- TRUNCATED
  S3   Paragraph              1,075   word        775       100%  <- TRUNCATED
  S4   Token-Exact-200        5,912  token        200         0%  <- OK
  S5   Section-200            6,153  token        180         0%  <- OK

B1 Pipeline (BGE-Large — 512 token limit)
  ID   Strategy              Chunks   Unit  AvgTokens     Trunc%
------------------------------------------------------------------------
  S1   Sliding-512w           1,431   word        781       100%  <- TRUNCATED
  S2   Sliding-256w           2,859   word        386       100%  <- TRUNCATED
  S3   Paragraph              1,075   word        775       100%  <- TRUNCATED
  S4   Token-Exact-400 

## Cell 8 — Synthetic QA Generation (Groq)
Generate 5 questions per category = 20 total from S4_200 chunks (cleanest text, no truncation).  
Same QA pairs reused for evaluating ALL strategies — ensures fair comparison.

In [9]:
client = OpenAI(
    api_key=OPENROUTER_API_KEY,
    base_url='https://openrouter.ai/api/v1',
)
r = client.chat.completions.create(
    model=GROQ_MODEL,
    messages=[{'role': 'user', 'content': 'Reply: ready'}],
    max_tokens=5,
)
print('OpenRouter:', r.choices[0].message.content.strip())


def generate_question(chunk_text, client):
    prompt = (
        'Read the financial/regulatory text and write ONE specific question '
        'that can only be answered using information in this text. '
        'Write as a compliance officer or financial analyst would ask it. '
        'Output ONLY the question.\n\nText:\n' + chunk_text[:600]
    )
    try:
        r = client.chat.completions.create(
            model=GROQ_MODEL,
            messages=[{'role': 'user', 'content': prompt}],
            max_tokens=60, temperature=0.3,
        )
        return r.choices[0].message.content.strip()
    except Exception as e:
        print('  QA error:', e)
        return None


QUESTIONS_PER_CAT = 5
np.random.seed(42)

qa_pairs = []
for cat in df_s4_200['category'].unique():
    cat_df  = df_s4_200[df_s4_200['category'] == cat]
    sampled = cat_df.sample(min(QUESTIONS_PER_CAT, len(cat_df)), random_state=42)
    print('Generating {} questions for [{}]...'.format(len(sampled), cat))
    for _, row in sampled.iterrows():
        q = generate_question(row['text'], client)
        time.sleep(0.5)
        if q:
            qa_pairs.append({
                'question':    q,
                'chunk_id':    row['chunk_id'],
                'chunk_text':  row['text'],
                'category':    cat,
                'source_file': row['source_file'],
            })
            print('  [{}] {}'.format(cat[:3], q[:80]))

print('\nTotal QA pairs:', len(qa_pairs))

# Save QA pairs
qa_path = os.path.join(RESULTS_DIR, 'synthetic_qa_pdf.json')
with open(qa_path, 'w') as f:
    json.dump(qa_pairs, f, indent=2)
print('Saved:', qa_path)

OpenRouter: I'm ready to assist
Generating 5 questions for [Regulatory]...
  [Reg] What specific regulatory section must an institution comply with before imposing
  [Reg] What timeframe is a remittance transfer provider required to correct an error af
  [Reg] What specific information is required to be disclosed by a financial institution
  [Reg] What timeframe is permitted for an institution to provide a receipt required by 
  [Reg] What specific condition must be met for the exclusion in 1005.20(b)(5) to not ap
Generating 5 questions for [Consumer_Protection]...
  [Con] What specific factors are considered when determining the order in which to pay 
  [Con] What statements are included in the self-assessment part 1 for evaluating one's 
  [Con] What specific information from an individual's credit history is typically inclu
  [Con] What specific organizational policies should be followed when storing and handli
  [Con] What type of fee is often waived if the cardholder signs up for 

## Cell 9 — Evaluation Helper: Self-Retrieval Recall@10
For each question: encode → search FAISS → check if any top-10 chunk contains the source text.  
Content-overlap check handles different chunk boundaries across strategies.

In [10]:
def build_faiss_index(df, model, prefix='', batch_size=128):
    """Encode chunk texts and build FAISS IndexFlatIP."""
    texts = df['text'].tolist()
    ids   = df['chunk_id'].tolist()
    id2text = dict(zip(ids, texts))
    embs = model.encode(
        texts, batch_size=batch_size,
        normalize_embeddings=True,
        show_progress_bar=True,
        convert_to_numpy=True,
    ).astype(np.float32)
    index = faiss.IndexFlatIP(embs.shape[1])
    index.add(embs)
    return index, ids, id2text


def content_recall_at_k(qa_pairs, index, chunk_ids, id2text, model,
                        query_prefix='', top_k=10):
    """
    Encode questions, search FAISS, check content overlap with source chunk.
    Uses first 80 chars of source chunk as fingerprint (handles different boundaries).
    Returns overall Recall@k and per-category breakdown.
    """
    questions   = [p['question']   for p in qa_pairs]
    src_texts   = [p['chunk_text'] for p in qa_pairs]
    categories  = [p['category']   for p in qa_pairs]

    prefixed = [query_prefix + q for q in questions] if query_prefix else questions
    q_embs   = model.encode(
        prefixed, normalize_embeddings=True,
        convert_to_numpy=True, show_progress_bar=False,
    ).astype(np.float32)

    D, I = index.search(q_embs, top_k)

    hits   = 0
    by_cat = {}
    for i, (src_text, cat) in enumerate(zip(src_texts, categories)):
        fingerprint = src_text[:80].lower().strip()
        retrieved   = [
            id2text.get(chunk_ids[idx], '').lower()
            for idx in I[i] if idx != -1 and idx < len(chunk_ids)
        ]
        found = any(fingerprint in rt for rt in retrieved)
        hits += int(found)
        by_cat.setdefault(cat, {'h': 0, 'n': 0})
        by_cat[cat]['n'] += 1
        by_cat[cat]['h'] += int(found)

    recall    = round(hits / len(qa_pairs), 4)
    cat_recall = {c: round(v['h']/v['n'], 4) for c, v in by_cat.items()}
    return recall, cat_recall


print('Evaluation helpers defined.')

Evaluation helpers defined.


## Cell 10 — A1 Pipeline: MiniLM × All 5 Strategies
Build 5 FAISS indexes with MiniLM. Evaluate Recall@10 for each strategy.  
No QE or Mistral reranking here — pure retrieval quality of each chunking strategy.

In [11]:
print('Loading MiniLM...')
minilm_model = SentenceTransformer(MINILM_NAME)
print('  Dim:', minilm_model.get_sentence_embedding_dimension())

a1_strategies = [
    ('S1', 'Sliding-512w',    df_s1),
    ('S2', 'Sliding-256w',    df_s2),
    ('S3', 'Paragraph',       df_s3),
    ('S4', 'Token-Exact-200', df_s4_200),
    ('S5', 'Section-200',     df_s5_200),
]

a1_results = {}
for sid, name, df in a1_strategies:
    print('\n[A1 + {}] Building index ({:,} chunks)...'.format(name, len(df)))
    index, ids, id2text_s = build_faiss_index(df, minilm_model)
    recall, by_cat = content_recall_at_k(
        qa_pairs, index, ids, id2text_s, minilm_model, query_prefix='', top_k=10
    )
    a1_results[sid] = {'name': name, 'recall': recall, 'by_cat': by_cat}
    print('[A1 + {}] Recall@10 = {}'.format(name, recall))
    # Free index memory
    del index
    gc.collect()

print('\n--- A1 (MiniLM) Summary ---')
for sid, v in a1_results.items():
    print('  {} {:<20} Recall@10 = {}'.format(sid, v['name'], v['recall']))

# Free MiniLM after A1 eval
del minilm_model
gc.collect()
print('\nMiniLM freed. Loading BGE-Large next...')

Loading MiniLM...


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 8248.62it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


  Dim: 384

[A1 + Sliding-512w] Building index (1,431 chunks)...


Batches: 100%|██████████| 12/12 [00:09<00:00,  1.28it/s]


[A1 + Sliding-512w] Recall@10 = 0.35

[A1 + Sliding-256w] Building index (2,859 chunks)...


...

[A1 + Paragraph] Recall@10 = 0.3

[A1 + Token-Exact-200] Building index (5,912 chunks)...


Batches: 100%|██████████| 47/47 [00:27<00:00,  1.74it/s]


[A1 + Token-Exact-200] Recall@10 = 0.75

[A1 + Section-200] Building index (6,153 chunks)...


Batches: 100%|██████████| 49/49 [00:27<00:00,  1.76it/s]


[A1 + Section-200] Recall@10 = 0.6

--- A1 (MiniLM) Summary ---
  S1 Sliding-512w         Recall@10 = 0.35
  S2 Sliding-256w         Recall@10 = 0.4
  S3 Paragraph            Recall@10 = 0.3
  S4 Token-Exact-200      Recall@10 = 0.75
  S5 Section-200          Recall@10 = 0.6

MiniLM freed. Loading BGE-Large next...


## Cell 11 — B1 Pipeline: BGE-Large × All 5 Strategies
Same evaluation with BGE-Large (1024-dim). Uses query prefix as required by BGE.  
S4_400 and S5_400 give BGE-Large full 400-token chunks (fits within 512-token limit).

In [12]:
print('Loading BGE-Large...')
bge_model = SentenceTransformer(BGELARGE_NAME)
print('  Dim:', bge_model.get_sentence_embedding_dimension())

b1_strategies = [
    ('S1', 'Sliding-512w',    df_s1),
    ('S2', 'Sliding-256w',    df_s2),
    ('S3', 'Paragraph',       df_s3),
    ('S4', 'Token-Exact-400', df_s4_400),
    ('S5', 'Section-400',     df_s5_400),
]

b1_results = {}
for sid, name, df in b1_strategies:
    print('\n[B1 + {}] Building index ({:,} chunks)...'.format(name, len(df)))
    # BGE-Large encodes CORPUS without prefix, QUERIES with prefix
    index, ids, id2text_s = build_faiss_index(df, bge_model, batch_size=64)
    recall, by_cat = content_recall_at_k(
        qa_pairs, index, ids, id2text_s, bge_model,
        query_prefix=BGE_PREFIX, top_k=10
    )
    b1_results[sid] = {'name': name, 'recall': recall, 'by_cat': by_cat}
    print('[B1 + {}] Recall@10 = {}'.format(name, recall))
    del index
    gc.collect()

print('\n--- B1 (BGE-Large) Summary ---')
for sid, v in b1_results.items():
    print('  {} {:<20} Recall@10 = {}'.format(sid, v['name'], v['recall']))

del bge_model
gc.collect()
print('\nBGE-Large freed.')

Loading BGE-Large...


Loading weights: 100%|██████████| 391/391 [00:00<00:00, 8247.20it/s]
BertModel LOAD REPORT from: BAAI/bge-large-en-v1.5
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


  Dim: 1024

[B1 + Sliding-512w] Building index (1,431 chunks)...


Batches: 100%|██████████| 23/23 [07:03<00:00, 18.42s/it]


[B1 + Sliding-512w] Recall@10 = 0.35

[B1 + Sliding-256w] Building index (2,859 chunks)...


...

[B1 + Paragraph] Recall@10 = 0.35

[B1 + Token-Exact-400] Building index (2,957 chunks)...


Batches: 100%|██████████| 47/47 [13:22<00:00, 17.07s/it]


[B1 + Token-Exact-400] Recall@10 = 0.8

[B1 + Section-400] Building index (3,368 chunks)...


Batches: 100%|██████████| 53/53 [14:40<00:00, 16.62s/it]


[B1 + Section-400] Recall@10 = 0.75

--- B1 (BGE-Large) Summary ---
  S1 Sliding-512w         Recall@10 = 0.35
  S2 Sliding-256w         Recall@10 = 0.4
  S3 Paragraph            Recall@10 = 0.35
  S4 Token-Exact-400      Recall@10 = 0.8
  S5 Section-400          Recall@10 = 0.75

BGE-Large freed.


## Cell 12 — Final Comparison Table + Winner

In [13]:
best_a1 = max(a1_results.values(), key=lambda x: x['recall'])
best_b1 = max(b1_results.values(), key=lambda x: x['recall'])
overall_best = max(
    [('A1', sid, v) for sid, v in a1_results.items()] +
    [('B1', sid, v) for sid, v in b1_results.items()],
    key=lambda x: x[2]['recall']
)

print()
print('=' * 80)
print('  FINAL: 5 STRATEGIES × 2 MODELS — Self-Retrieval Recall@10')
print('  Corpus: 4 selected PDFs (1 per category)  |  Queries: 20 synthetic QA pairs')
print('=' * 80)
print('  {:<4} {:<7} {:<22} {:>10} {:>12}'.format(
    'ID', 'Model', 'Strategy', 'Recall@10', 'Truncation'))
print('-' * 80)

trunc_map = {'S1': '100%', 'S2': '100%', 'S3': '100%', 'S4': '0%', 'S5': '0%'}

all_rows = []
for sid, v in a1_results.items():
    all_rows.append(('A1', 'MiniLM', sid, v['name'], v['recall'], trunc_map[sid]))
for sid, v in b1_results.items():
    all_rows.append(('B1', 'BGE-L', sid, v['name'], v['recall'], trunc_map[sid]))

best_recall = max(r[4] for r in all_rows)
for pipe, model, sid, name, recall, trunc in all_rows:
    tag = '  <- BEST' if recall == best_recall else ''
    print('  {:<4} {:<7} {:<22} {:>10.4f} {:>12}{}'.format(
        sid, pipe+'/'+model, name, recall, trunc, tag))

print('=' * 80)
print()
print('Winner: {} + {} — {} (Recall@10 = {})'.format(
    overall_best[0], overall_best[1], overall_best[2]['name'], overall_best[2]['recall']))
print()
print('Key findings:')
# S4 vs S1 for A1
if 'S4' in a1_results and 'S1' in a1_results:
    delta_a1 = round(a1_results['S4']['recall'] - a1_results['S1']['recall'], 4)
    print('  A1: Token-exact (S4) vs Word-based (S1): delta = {} ({})'.format(
        delta_a1, 'S4 better' if delta_a1 > 0 else 'S1 better'))
if 'S4' in b1_results and 'S1' in b1_results:
    delta_b1 = round(b1_results['S4']['recall'] - b1_results['S1']['recall'], 4)
    print('  B1: Token-exact (S4) vs Word-based (S1): delta = {} ({})'.format(
        delta_b1, 'S4 better' if delta_b1 > 0 else 'S1 better'))
if 'S5' in a1_results and 'S4' in a1_results:
    delta_s5 = round(a1_results['S5']['recall'] - a1_results['S4']['recall'], 4)
    print('  A1: Section-aware (S5) vs Token-exact (S4): delta = {} ({})'.format(
        delta_s5, 'S5 better' if delta_s5 > 0 else 'S4 better'))
if best_a1['recall'] > best_b1['recall']:
    print('  MiniLM beats BGE-Large on this PDF corpus (smaller model, better fit)')
else:
    print('  BGE-Large beats MiniLM on this PDF corpus (larger model wins)')

# Save results
out = {
    'corpus': '4 selected PDFs (1 per category)',
    'qa_pairs': len(qa_pairs),
    'A1_MiniLM': {sid: {'strategy': v['name'], 'recall_10': v['recall'],
                         'by_category': v['by_cat']} for sid, v in a1_results.items()},
    'B1_BGELarge': {sid: {'strategy': v['name'], 'recall_10': v['recall'],
                           'by_category': v['by_cat']} for sid, v in b1_results.items()},
    'winner': {
        'pipeline': overall_best[0],
        'strategy': overall_best[1],
        'name':     overall_best[2]['name'],
        'recall_10': overall_best[2]['recall'],
    }
}
out_path = os.path.join(RESULTS_DIR, 'chunk_eval_results.json')
with open(out_path, 'w') as f:
    json.dump(out, f, indent=2)
print('\nSaved:', out_path)
print('Winner chunk strategy feeds into: intent classification notebook next.')


  FINAL: 5 STRATEGIES × 2 MODELS — Self-Retrieval Recall@10
  Corpus: 4 selected PDFs (1 per category)  |  Queries: 20 synthetic QA pairs
  ID   Model   Strategy                Recall@10   Truncation
--------------------------------------------------------------------------------
  S1   A1/MiniLM Sliding-512w               0.3500         100%
  S2   A1/MiniLM Sliding-256w               0.4000         100%
  S3   A1/MiniLM Paragraph                  0.3000         100%
  S4   A1/MiniLM Token-Exact-200            0.7500           0%
  S5   A1/MiniLM Section-200                0.6000           0%
  S1   B1/BGE-L Sliding-512w               0.3500         100%
  S2   B1/BGE-L Sliding-256w               0.4000         100%
  S3   B1/BGE-L Paragraph                  0.3500         100%
  S4   B1/BGE-L Token-Exact-400            0.8000           0%  <- BEST
  S5   B1/BGE-L Section-400                0.7500           0%

Winner: B1 + S4 — Token-Exact-400 (Recall@10 = 0.8)

Key findings:
  A1: 